# 04 - Combined dataset

Composes destinations + Ryanair fares + the on-the-ground cost basket +
accommodation anchors into per-destination records the frontend uses directly.

For each destination, computes:
- Anchor airport (Ryanair-served, if any) and a `no_ryanair_route` flag
- A `routes` block per origin: daily outbound/return fare calendars, plus the
  ground-transport transfer for gems (airport to town)
- A `costs` block: the lifestyle price basket for that location (city override
  where we have one, otherwise the country basket)
- An `accommodation` block: the Airbnb nightly anchor (city override else country)

Writes `cache/combined_dataset.json`.

## 1. Setup + load everything

In [1]:
import json
from datetime import date, datetime, timezone
from pathlib import Path

CACHE_DIR = Path("cache")
cfg     = json.loads((CACHE_DIR / "config.json").read_text(encoding="utf-8"))
master  = json.loads((CACHE_DIR / "destinations_master.json").read_text(encoding="utf-8"))
fares   = json.loads((CACHE_DIR / "fare_calendars.json").read_text(encoding="utf-8"))
costs   = json.loads((CACHE_DIR / "costs.json").read_text(encoding="utf-8"))
accom   = json.loads((CACHE_DIR / "accommodation.json").read_text(encoding="utf-8"))

destinations    = master["destinations"]
airports        = master["airports"]
fare_calendars  = fares["fare_calendars"]
cost_countries  = costs["countries"]
cost_cities     = costs["cities"]
COST_ITEMS      = costs["meta"]["items"]
accom_countries = accom["countries"]
accom_cities    = accom["cities"]
ACCOM_FIELDS    = ["per_person_night_eur", "cleaning_per_person_eur",
                   "entire_home_night_eur", "typical_capacity"]

print(f"Destinations: {len(destinations)}")
print(f"Fare routes:  {sum(len(v) for v in fare_calendars.values())}")
print(f"Cost baskets: {len(cost_countries)} countries, {len(cost_cities)} cities")
print(f"Accom anchors: {len(accom_countries)} countries, {len(accom_cities)} cities")

Destinations: 450
Fare routes:  512
Cost baskets: 42 countries, 22 cities
Accom anchors: 42 countries, 32 cities


## 2. Resolve anchor airport per destination (Ryanair-served only)

In [2]:
def resolve_anchor(d):
    """Return (anchor_iata, ground_minutes, ground_eur_one_way) or None."""
    if d["tier"] == "airport":
        if d.get("ryanair_serves"):
            return (d["iata"], 0, 0)
        return None
    for iata, mins, eur in d["nearest_airports"]:
        a = airports.get(iata)
        if a and a.get("ryanair_serves"):
            return (iata, mins, eur)
    return None

anchors = {}
no_route_count = 0
for d in destinations:
    anchor = resolve_anchor(d)
    if anchor is None:
        no_route_count += 1
    else:
        anchors[d["id"]] = anchor

print(f"Routed: {len(anchors)} / {len(destinations)}")
print(f"No Ryanair route: {no_route_count}")

Routed: 439 / 450
No Ryanair route: 11


## 3. Transfer warning logic for gems

A gem gets `airport_transfer_warning` if:
- Ground transport > 60 minutes one-way, OR
- Ground transport cost > €15 one-way per person, OR
- A closer non-Ryanair airport exists but is unused (worth telling the user)

In [3]:
def transfer_warning(d, anchor):
    if d["tier"] != "gem" or anchor is None:
        return None
    anchor_iata, mins, eur = anchor

    msgs = []
    if mins > 60:
        msgs.append(f"~{mins} min from {anchor_iata}")
    if eur > 15:
        msgs.append(f"€{eur:.0f}/person one-way")

    # Is there a closer non-Ryanair option that the user might prefer to book elsewhere?
    closer_alternatives = []
    for alt_iata, alt_mins, alt_eur in d["nearest_airports"]:
        if alt_iata == anchor_iata:
            continue
        if alt_mins < mins - 15:   # meaningfully closer
            a = airports.get(alt_iata, {})
            label = a.get("city", alt_iata)
            closer_alternatives.append(f"{alt_iata} ({label}, ~{alt_mins} min)")

    if not msgs and not closer_alternatives:
        return None

    return {
        "transfer_minutes_one_way":  mins,
        "transfer_eur_one_way_pp":   eur,
        "summary":                   "; ".join(msgs) if msgs else "long transfer",
        "closer_alternatives":       closer_alternatives[:2],
    }

## 4. Compose dataset

In [4]:
def route_fares(origin, anchor_iata):
    """Flatten the fetched calendar into {date: cheapest_eur} for each direction."""
    r = fare_calendars.get(origin, {}).get(anchor_iata, {})
    out = {d: v["cheapest"] for d, v in r.get("out", {}).items()}
    ret = {d: v["cheapest"] for d, v in r.get("ret", {}).items()}
    return out, ret

def costs_for(d):
    """Lifestyle price basket for a destination: city override, else country.
    City names carry airport qualifiers (e.g. 'Rome (Fiumicino)'); strip those
    before matching a city override."""
    city_key = d["city"].split(" (")[0].strip()
    city = cost_cities.get(city_key)
    level, basket = ("city", city) if city else ("country", cost_countries.get(d["iso2"]))
    if not basket:
        return None
    out = {k: basket[k] for k in COST_ITEMS}
    out["level"] = level                    # "city" or "country"
    out["price_source"] = basket["source"]  # numbeo_city / numbeo_direct / pli_scaled
    return out

def accom_for(d):
    """Airbnb nightly anchor for a destination: city override, else country.
    Same city-name matching as costs_for."""
    city_key = d["city"].split(" (")[0].strip()
    city = accom_cities.get(city_key)
    level, a = ("city", city) if city else ("country", accom_countries.get(d["iso2"]))
    if not a:
        return None
    out = {k: a[k] for k in ACCOM_FIELDS}
    out["level"] = level                    # "city" or "country"
    out["price_source"] = a["source"]       # inside_airbnb_city / _country / airbnb_pli_scaled
    return out

dataset = {}
for d in destinations:
    did    = d["id"]
    anchor = anchors.get(did)

    # Build per-origin route blocks (only where the anchor has fares)
    routes = {}
    if anchor is not None:
        anchor_iata, ground_mins, ground_eur = anchor
        for origin in cfg["origins"]:
            out, ret = route_fares(origin, anchor_iata)
            if not out and not ret:
                continue
            routes[origin] = {
                "anchor_airport":               anchor_iata,
                "ground_transport_one_way_eur": ground_eur,
                "ground_transport_minutes":     ground_mins,
                "outbound_fare":                out,
                "return_fare":                  ret,
            }

    dataset[did] = {
        "id":         did,
        "tier":       d["tier"],
        "iata":       d.get("iata"),
        "city":       d["city"],
        "country":    d["country"],
        "iso2":       d["iso2"],
        "lat":        d["lat"],
        "lon":        d["lon"],
        "categories": d.get("categories", []),
        "tags":       d.get("tags", []),
        "blurb":      d.get("blurb"),

        # Ryanair routing
        "no_ryanair_route": anchor is None,
        "anchor_airport":   anchor[0] if anchor else None,
        "transfer":         transfer_warning(d, anchor),
        "routes":           routes,

        # On-the-ground cost basket (lifestyle pricing)
        "costs":            costs_for(d),

        # Accommodation (Airbnb nightly anchor)
        "accommodation":    accom_for(d),
    }

n_city  = sum(1 for r in dataset.values() if r["costs"] and r["costs"]["level"] == "city")
n_acity = sum(1 for r in dataset.values() if r["accommodation"] and r["accommodation"]["level"] == "city")
print(f"Built {len(dataset)} records")
print(f"  with Ryanair route:    {sum(1 for r in dataset.values() if not r['no_ryanair_route'])}")
print(f"  with fare data:        {sum(1 for r in dataset.values() if r['routes'])}")
print(f"  with cost basket:      {sum(1 for r in dataset.values() if r['costs'])}  ({n_city} use city-level prices)")
print(f"  with accommodation:    {sum(1 for r in dataset.values() if r['accommodation'])}  ({n_acity} use city-level rates)")
print(f"  with transfer warning: {sum(1 for r in dataset.values() if r['transfer'])}")

Built 450 records
  with Ryanair route:    439
  with fare data:        177
  with cost basket:      450  (24 use city-level prices)
  with accommodation:    450  (37 use city-level rates)
  with transfer warning: 126


## 5. Save

In [5]:
OUT = CACHE_DIR / "combined_dataset.json"
payload = {
    "meta": {
        "generated_at":   datetime.now(timezone.utc).isoformat(),
        "schema_version": cfg["schema_version"],
        "n_destinations": len(dataset),
    },
    "destinations": dataset,
}
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {OUT}  ({OUT.stat().st_size/1024:.1f} KB)")

Wrote cache\combined_dataset.json  (594.9 KB)


## 6. Spot-check one destination

In [6]:
for test_id in ["MAD", "FCO", "CDG", "gem:bruges"]:
    if test_id not in dataset:
        continue
    r = dataset[test_id]
    print(f"\n--- {test_id}: {r['city']}, {r['country']} ({r['tier']}) ---")
    print(f"  Anchor: {r['anchor_airport']}  no_ryanair: {r['no_ryanair_route']}")
    bru = r["routes"].get("BRU", {})
    out_cal = bru.get("outbound_fare", {})
    if out_cal:
        print(f"  Fares BRU outbound: {len(out_cal)} days, min EUR {min(out_cal.values()):.2f}")
    c = r["costs"]
    if c:
        print(f"  Costs ({c['level']}, {c['price_source']}): dinner EUR {c['meal_mid_eur']:.2f} | "
              f"drink EUR {c['drink_out_eur']:.2f} | coffee EUR {c['coffee_eur']:.2f} | "
              f"grocery/day EUR {c['grocery_day_eur']:.2f}")
    a = r["accommodation"]
    if a:
        print(f"  Accom ({a['level']}, {a['price_source']}): EUR {a['entire_home_night_eur']:.0f}/night whole home "
              f"(cap {a['typical_capacity']}) = EUR {a['per_person_night_eur']:.2f} pp/night "
              f"+ EUR {a['cleaning_per_person_eur']:.2f} clean")


--- MAD: Madrid, Spain (airport) ---
  Anchor: MAD  no_ryanair: False
  Fares BRU outbound: 4 days, min EUR 29.99
  Costs (city, numbeo_city): dinner EUR 30.00 | drink EUR 3.50 | coffee EUR 2.81 | grocery/day EUR 11.62
  Accom (city, inside_airbnb_city): EUR 120/night whole home (cap 4) = EUR 30.00 pp/night + EUR 15.00 clean

--- FCO: Rome (Fiumicino), Italy (airport) ---
  Anchor: FCO  no_ryanair: False
  Fares BRU outbound: 4 days, min EUR 29.99
  Costs (country, numbeo_direct): dinner EUR 35.00 | drink EUR 5.00 | coffee EUR 1.76 | grocery/day EUR 13.00
  Accom (city, inside_airbnb_city): EUR 130/night whole home (cap 4) = EUR 32.50 pp/night + EUR 16.25 clean

--- CDG: Paris (CDG), France (airport) ---
  Anchor: CDG  no_ryanair: False
  Costs (country, numbeo_direct): dinner EUR 30.00 | drink EUR 6.02 | coffee EUR 3.45 | grocery/day EUR 12.65
  Accom (city, inside_airbnb_city): EUR 160/night whole home (cap 4) = EUR 40.00 pp/night + EUR 20.00 clean

--- gem:bruges: Bruges, Belgium (